In [3]:
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q
!pip install databricks-sql-connector pandas -q
!pip install yfinance -q

In [4]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

Mounted at /content/drive
✅ Drive mounted successfully


In [5]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [6]:
import boto3
import os
import glob
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import date, datetime
from pyspark.sql import DataFrame
from functools import reduce
import re

# ── boto3 Client ──────────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

# ── Download All Partitioned Files from MinIO ─────────────────────────────────
local_dir  = "/tmp/hlebroking_Portfolio"
bucket     = "rawdatasets"
prefix     = "hlebroking_Portfolio/"

os.makedirs(local_dir, exist_ok=True)

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    s3_key = obj["Key"]

    # Skip if the key is just the folder prefix itself
    if s3_key.endswith("/"):
        continue

    # Reconstruct local path preserving year=/month= structure
    relative_path = os.path.relpath(s3_key, prefix)
    local_file_path = os.path.join(local_dir, relative_path)

    # Create subdirectories if needed (e.g. year=2023/month=6/)
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    s3.download_file(Bucket=bucket, Key=s3_key, Filename=local_file_path)
    #print(f"Downloaded: {s3_key} → {local_file_path}")

# ── Read All Partitions into Spark DataFrame ──────────────────────────────────
spark = SparkSession.builder \
    .appName("ReadCSGCSV") \
    .getOrCreate()

dfs = []

for filename in os.listdir(local_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(local_dir, filename)

        # Get file metadata
        year, month, day = None, None, None  # defaults

        match = re.search(r'_?(\d{4})(\d{2})(\d{2})', filename)
        if match:
            year, month, day = match.groups()

        if year and month and day:
            created_time = date(int(year), int(month), int(day))
            date_created = created_time.strftime("%Y-%m-%d")
        else:
            created_time = date.today()  # or a fallback date, e.g. datetime.date.today()
            date_created = created_time.strftime("%Y-%m-%d")

        #created_time = os.path.getctime(filepath)
        #date_created = datetime.fromtimestamp(created_time).strftime("%Y-%m-%d")

        # Read each CSV and append metadata columns
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(filepath)

        df = df.withColumn("source_file", F.lit(filename)) \
               .withColumn("date_created", F.lit(date_created))

        dfs.append(df)

# Union all DataFrames into one
final_df = reduce(DataFrame.union, dfs)

#final_df.printSchema()
#final_df.show()

In [7]:
import re

def clean_column_names(df):
    new_columns = []
    for col in df.columns:
        # Remove special characters, keep only alphanumeric and underscores
        cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', col)
        # Remove leading/trailing underscores
        cleaned = cleaned.strip('_')
        # Replace multiple consecutive underscores with single one
        cleaned = re.sub(r'_+', '_', cleaned)
        new_columns.append(cleaned)

    # Rename all columns
    for old, new in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old, new)

    return df

def rename_duplicates(df):
    seen = {}
    new_cols = []
    for col in df.columns:
        count = df.columns.count(col)
        if count > 1:
            seen[col] = seen.get(col, 0) + 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            new_cols.append(col)
    return df.toDF(*new_cols)

# Apply to your DataFrame
final_df = clean_column_names(final_df)
final_df = rename_duplicates(final_df)
#final_df.printSchema()
#final_df.show()

In [8]:
#Modify column data
final_df = final_df.withColumn(
    "Stock_code",
    F.when(F.col("Stock_code").isNull(), F.col("Stock_code"))
     .otherwise(F.concat(F.trim(F.col("Stock_code").cast("string")), F.lit(".KL")))
).withColumn("Qty" , F.regexp_replace(F.col("Qty"), ",", "").cast("int")
).withColumn("Gross_Investment" , F.regexp_replace(F.col("Gross_Investment"), ",", "").cast("decimal(18, 4)")
).withColumn("Market_Value" , F.regexp_replace(F.col("Market_Value"), ",", "").cast("decimal(18, 4)")
).withColumn("Unrealized_P_L_1" , F.regexp_replace(F.col("Unrealized_P_L_1"), ",", "").cast("decimal(18, 4)"))
#final_df.printSchema()
#final_df.show()

In [9]:
import yfinance as yf
import pandas as pd

# Decode bytes → string → wrap in StringIO for pandas
pdf = final_df.toPandas()

def get_info(symbol):
    try:
        info = yf.Ticker(symbol).info
        return pd.Series({
            "name": info.get("longName", "N/A"),
            "current_price": info.get("currentPrice", info.get("regularMarketPrice", 0)),
            "shortName": info.get("shortName", "N/A"),
            "sector": info.get("sector", "N/A"),
            "industry": info.get("industry", "N/A"),
            "country": info.get("country", "N/A"),
            "city": info.get("city", "N/A"),
            "state": info.get("state", "N/A"),
            "website": info.get("website", "N/A"),
            "marketCap": round(float(info.get("marketCap")) / 1_000_000, 2) if info.get("marketCap") else 0,
            "enterpriseToRevenue": info.get("enterpriseToRevenue", 0),
            "dividendYield": info.get("dividendYield", 0),
            "dividendRate": info.get("dividendRate", 0),
            "payoutRatio": info.get("payoutRatio", 0),
            "exDividendDate": pd.to_datetime(info.get("exDividendDate"), unit="s").date() if info.get("exDividendDate") else None,
            "lastDividendDate": pd.to_datetime(info.get("lastDividendDate"), unit="s").date() if info.get("lastDividendDate") else None,
            "lastDividendValue": info.get("lastDividendValue", 0),
            "lastSplitFactor": info.get("lastSplitFactor", "N/A"),
            "lastSplitDate": pd.to_datetime(info.get("lastSplitDate"), unit="s").date() if info.get("lastSplitDate") else None,
        })
    except:
        return pd.Series({"name": "N/A", "current_price": None})

# Apply row-wise — adds 2 new columns directly
pdf[["name", "current_price", "shortName", "sector", "industry", "country"
, "city", "state", "website", "marketCap", "enterpriseToRevenue", "dividendYield"
, "dividendRate", "payoutRatio" , "exDividendDate", "lastDividendDate", "lastDividendValue"
, "lastSplitFactor", "lastSplitDate"
]] = pdf["Stock_code"].apply(get_info)

df = spark.createDataFrame(pdf)

df.printSchema()
df.show()

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 3395WD.KL"}}}


root
 |-- Client_Code: string (nullable = true)
 |-- Stock_code: string (nullable = true)
 |-- Stock_Name: string (nullable = true)
 |-- Qty: long (nullable = true)
 |-- Exchange: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Avg_Price: double (nullable = true)
 |-- Market_Price: double (nullable = true)
 |-- Gross_Investment: decimal(38,18) (nullable = true)
 |-- Market_Value: decimal(38,18) (nullable = true)
 |-- Unrealized_P_L_1: decimal(38,18) (nullable = true)
 |-- Unrealized_P_L_2: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- date_created: string (nullable = true)
 |-- name: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- shortName: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- website: string (nullable = true)
 |-- marketCap: doub

In [10]:
# ─── Databricks Connection Config ───────────────────────────────────────────
SERVER_HOSTNAME = G_SERVER_HOSTNAME
HTTP_PATH       = G_HTTP_PATH   # from Connection Details tab
ACCESS_TOKEN    = G_ACCESS_TOKEN         # replace with your PAT

# ─── Catalog / Schema / Table Target ────────────────────────────────────────
CATALOG   = "workspace"            # change to your catalog name (Unity Catalog)
SCHEMA    = "default"         # change to your schema/database
TABLE     = "hlebroking_portfolio_yahoo"
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [11]:
from databricks import sql

def run_query(cursor, query):
    """Helper to execute and optionally fetch results."""
    cursor.execute(query)
    try:
        return cursor.fetchall()
    except Exception:
        return None

COLUMNS = [
    "Client_Code",
    "Stock_code",
    "Stock_Name",
    "Qty",
    "Exchange",
    "Currency",
    "Avg_Price",
    "Market_Price",
    "Gross_Investment",
    "Market_Value",
    "Unrealized_P_L_1",
    "Unrealized_P_L_2",
    "source_file",
    "date_created",
    "name",
    "current_price",
    "shortName",
    "sector",
    "industry",
    "country",
    "city",
    "state",
    "website",
    "marketCap",
    "enterpriseToRevenue",
    "dividendYield",
    "dividendRate",
    "payoutRatio",
    "exDividendDate",
    "lastDividendDate",
    "lastDividendValue",
    "lastSplitFactor",
    "lastSplitDate"
]

placeholders = ", ".join(["?"] * len(COLUMNS))
col_names    = ", ".join(COLUMNS)

with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        # ── Step 1: Set catalog and schema context ───────────────────────────
        cursor.execute(f"USE CATALOG {CATALOG}")
        cursor.execute(f"USE SCHEMA {SCHEMA}")
        cursor.execute(f"DROP TABLE IF EXISTS {FULL_TABLE}")

        # ── Step 2: Create table if not exists ───────────────────────────────
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
              Client_Code         STRING,
              Stock_code          STRING,
              Stock_Name          STRING,
              Qty                 DOUBLE,
              Exchange            STRING,
              Currency            STRING,
              Avg_Price           DOUBLE,
              Market_Price        DOUBLE,
              Gross_Investment    DOUBLE,
              Market_Value        DOUBLE,
              Unrealized_P_L_1    DOUBLE,
              Unrealized_P_L_2    DOUBLE,
              source_file         STRING,
              date_created        STRING,
              name                STRING,
              current_price       DOUBLE,
              shortName           STRING,
              sector              STRING,
              industry            STRING,
              country             STRING,
              city                STRING,
              state               STRING,
              website             STRING,
              marketCap           DOUBLE,
              enterpriseToRevenue DOUBLE,
              dividendYield       DOUBLE,
              dividendRate        DOUBLE,
              payoutRatio         DOUBLE,
              exDividendDate      DATE,
              lastDividendDate    DATE,
              lastDividendValue   DOUBLE,
              lastSplitFactor     STRING,
              lastSplitDate       DATE
        )
        """
        cursor.execute(create_sql)
        print(f"✅ Table '{FULL_TABLE}' created/verified.")

        # ── Step 3: Insert rows from Pandas DataFrame ─────────────────────────

        rows = [tuple(row) for row in df.collect()]


        insert_sql = f"""
        INSERT INTO {FULL_TABLE} ({col_names})
        SELECT {placeholders}
        WHERE NOT EXISTS (
            SELECT 1 FROM {FULL_TABLE}
            WHERE Stock_code = ?
              AND Client_Code = ?
              AND source_file = ?
        )
        """


        # Append the 3 key values at end of each row for the WHERE NOT EXISTS check
        rows_with_keys = [
            row + (row[COLUMNS.index("Stock_code")],
                  row[COLUMNS.index("Client_Code")],
                  row[COLUMNS.index("source_file")])
            for row in rows
        ]

        cursor.executemany(insert_sql, rows_with_keys)

        print(f"✅ Inserted {df.count()} rows into '{FULL_TABLE}'.")

        # ── Step 4: Verify by reading back ───────────────────────────────────
        results = run_query(cursor, f"SELECT * FROM {FULL_TABLE}")
        print(f"\n📦 Data in '{FULL_TABLE}':")
        for r in results:
            print(r)

✅ Table 'workspace.default.hlebroking_portfolio_yahoo' created/verified.
✅ Inserted 25 rows into 'workspace.default.hlebroking_portfolio_yahoo'.

📦 Data in 'workspace.default.hlebroking_portfolio_yahoo':
Row(Client_Code='LA1110', Stock_code='5127.KL', Stock_Name='ARREIT', Qty=1000.0, Exchange='Bursa', Currency='MYR', Avg_Price=0.7300000190734863, Market_Price=0.3100000023841858, Gross_Investment=730.0, Market_Value=310.0, Unrealized_P_L_1=-420.0, Unrealized_P_L_2=-57.529998779296875, source_file='hlebroking_20260305.csv', date_created='2026-03-05', name='AmanahRaya Real Estate Investment Trust', current_price=0.3149999976158142, shortName='ARREIT', sector='Real Estate', industry='REIT - Diversified', country='Malaysia', city='Kuala Lumpur', state='N/A', website='https://www.amanahrayareit.com.my', marketCap=180.55999755859375, enterpriseToRevenue=8.491000175476074, dividendYield=2.3299999237060547, dividendRate=0.009999999776482582, payoutRatio=1.8458000421524048, exDividendDate=dateti

In [12]:
spark.stop()